# Loans at Risk: Capturing Default — Extract · Transform · Load

## Purpose

This notebook implements the ETL layer of the LendingClub default prediction project.  
The training window covers **2007–2014**; the test window covers **2015**.

The objective is not analysis. It is structural control.

Raw CSV extracts are converted into datasets that are suitable for downstream modeling under a clearly defined constraint: prediction is made at the moment of application submission.

“Structurally reliable” in this context means:

- Identical schema across train and test  
- Explicit treatment of datatypes and null structure  
- Removal of null-only and constant columns  
- Elimination of export artifacts  
- Early enforcement of the submission-time prediction boundary  

No interpretation is performed here. This notebook establishes a clean, governed input layer so that later analytical steps operate on stable ground.
All dependency management is handled at repository level (`README`, `requirements.txt`). This notebook assumes the environment is already configured.

---

## Extract

Raw LendingClub loan data is ingested directly from the source CSV files.

The two temporal partitions are kept strictly separate:

- **2007–2014** → training  
- **2015** → testing  

Extraction preserves the raw state of the data. No filtering, type coercion, or feature selection occurs at this stage. The raw layer is treated as immutable input.

---

## Transform

The transformation phase enforces structural and temporal discipline.

Concretely:

- Schema and datatype validation  
- Identification and removal of null-only and constant columns  
- Detection of mixed types and object-encoded numerics  
- Removal of structural artifacts (e.g., exported index columns)  
- Explicit exclusion of post-origination, underwriting, and pricing variables  
- Deterministic type conversions (including numeric and date normalization)  
- Structured handling of credit-event timing variables, where null represents absence rather than data loss  
- Identical transformation logic across training and test sets  

No feature engineering beyond structural normalization is performed.  
No modeling assumptions are introduced.

The output of this phase is a submission-time compliant dataset.

---

## Load

The transformed datasets are persisted in Parquet format.

Two layers are materialized:

- **Clean dataset** → structurally aligned variables  
- **Feature-base dataset** → submission-time eligible variables plus target  

Schema identity between training and test is validated prior to persistence.
These outputs form the controlled input layer for EDA and modeling.

---

## Scope Boundary

Feature engineering, modeling, evaluation, and decision logic are handled in subsequent notebooks.
This notebook exists to ensure that those stages operate on data that is structurally sound, temporally consistent, and reproducible.

In [1]:
from pathlib import Path
import sys

current_path = Path.cwd().resolve()
project_root = None

for parent_path in (current_path, *current_path.parents):
    if (parent_path / "pyproject.toml").exists():
        project_root = parent_path
        break

if project_root is None:
    raise RuntimeError(
        f"Failed to resolve project root: pyproject.toml not found from {current_path}"
    )

src_path = project_root / "src"
if not src_path.exists():
    raise RuntimeError(f"Expected 'src' directory at: {src_path}")

if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

{
    "stage": "bootstrap_import_path_ready",
    "project_root": str(project_root),
}

{'stage': 'bootstrap_import_path_ready',
 'project_root': 'D:\\Portfolio\\loans-at-risk-capturing-default'}

In [2]:
# -------------------------------
# Imports: libraries and custom functions
# -------------------------------

from datetime import datetime, timezone
from typing import Callable
import uuid

import pandas as pd

# Editable-installed local package
from config.logging import (
    get_logger,
    log_messages,
    log_dataframe_checkpoint,
)
from dataset.preprocessing import (
    apply_categorical_mapping,
    apply_credit_sentinel_encoding,
    convert_month_year_columns_to_datetime,
    create_row_identifier,
    drop_columns_with_logging,
    initial_inspection,
    normalize_home_ownership,
    normalize_text_columns,
    transform_emp_length,
    build_combined_schema,
    build_structural_issues_report,
    build_string_alignment_report,
    parse_term_column,
    cast_categorical_columns,
)

In [3]:
# ===============================
# Paths and run context
# ===============================

logs_root = project_root / "logs"
logs_root.mkdir(parents=True, exist_ok=True)

PROJECT_LOG_FILE = logs_root / "project.log"

run_id = uuid.uuid4().hex[:10]
run_timestamp_utc = datetime.now(timezone.utc)

run_header = (
    "NEW RUN: "
    f"{run_timestamp_utc.strftime('%Y-%m-%d %H:%M:%S')} UTC | "
    f"RUN_ID={run_id}"
)

log_messages("\n" + "=" * 60, PROJECT_LOG_FILE)
log_messages(run_header, PROJECT_LOG_FILE)
log_messages("=" * 60 + "\n", PROJECT_LOG_FILE)

log: Callable[[str], None] = get_logger(PROJECT_LOG_FILE)
log("ETL notebook initialized")
log(f"project_root: {project_root}")

log(run_header)

{
    "stage": "run_started",
    "run_id": run_id,
    "utc_timestamp": run_timestamp_utc.isoformat(),
}


# ---------------------------------------------------------------
# Inputs for this notebook (raw, interim, report)
# ---------------------------------------------------------------

raw_train_data_file = project_root / "data" / "external" / "loan_data_2007_2014.csv"
raw_test_data_file = project_root / "data" / "external" / "loan_data_2015.csv"

clean_train_data_file = project_root / "data" / "interim" / "clean_loan_data_2007_2014.parquet"
clean_test_data_file = project_root / "data" / "interim" / "clean_loan_data_2015.parquet"

feature_base_train_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2007_2014.parquet"
feature_base_test_data_file = project_root / "data" / "interim" / "feature_base_loan_data_2015.parquet"

audit_dir = project_root / "artifacts" / "audit"

appendix_base_name = "appendix_A_raw_feature_universe"

appendix_parquet_file = audit_dir / f"{appendix_base_name}.parquet"
appendix_csv_file = audit_dir / f"{appendix_base_name}.csv"

required_directories = {
    raw_train_data_file.parent,
    raw_test_data_file.parent,
    clean_train_data_file.parent,
    clean_test_data_file.parent,
    feature_base_train_data_file.parent,
    feature_base_test_data_file.parent,
    appendix_parquet_file.parent,
    appendix_csv_file.parent,
}

for directory_path in sorted(required_directories):
    directory_path.mkdir(parents=True, exist_ok=True)

log(f"Raw training data path: {raw_train_data_file}")
log(f"Raw test data path: {raw_test_data_file}")
log(f"Clean training parquet path: {clean_train_data_file}")
log(f"Clean test parquet path: {clean_test_data_file}")
log(f"Feature base training parquet path: {feature_base_train_data_file}")
log(f"Feature base test parquet path: {feature_base_test_data_file}")
log(f"Appendix parquet path: {appendix_parquet_file}")
log(f"Appendix CSV path: {appendix_csv_file}")

## Initial Data Inspection

This step examines the raw dataset exactly as delivered.

The objective is to map its structural characteristics before any transformation rules are applied.

The inspection evaluates:

- Column count and schema alignment  
- Datatype assignments and object-encoded numerics  
- Fully null and constant columns  
- Patterns of missingness, including variables where null reflects structural absence  
- Mixed-type inconsistencies  
- Identifier fields and exported artifacts  

No variables are removed and no values are altered at this stage.

Findings from this inspection determine which columns require exclusion, normalization, or explicit encoding during transformation. Every later modification is grounded in observations documented here.

In [4]:
# --------------------------------------------------------
# Load raw datasets (train/test) + checkpoint
# --------------------------------------------------------

df_raw_train_data = pd.read_csv(raw_train_data_file, low_memory=False)
df_raw_test_data = pd.read_csv(raw_test_data_file, low_memory=False)

raw_overview_df = pd.DataFrame(
    [
        {
            "dataset": "train_raw",
            "rows": df_raw_train_data.shape[0],
            "cols": df_raw_train_data.shape[1],
            "memory_mb": round(df_raw_train_data.memory_usage(deep=True).sum() / (1024**2), 2),
        },
        {
            "dataset": "test_raw",
            "rows": df_raw_test_data.shape[0],
            "cols": df_raw_test_data.shape[1],
            "memory_mb": round(df_raw_test_data.memory_usage(deep=True).sum() / (1024**2), 2),
        },
    ]
)

display(raw_overview_df)
log(f"[raw] overview: {raw_overview_df.to_dict(orient='records')}")

,dataset,rows,cols,memory_mb
0,train_raw,466285,75,389.20
1,test_raw,421094,74,322.85


In [5]:
# -----------------------------------------
# Raw schema comparison (train vs test)
# -----------------------------------------

combined_schema_raw = build_combined_schema(
    train_dataframe=df_raw_train_data,
    test_dataframe=df_raw_test_data,
    stage_name="raw",
    log=log,
)

# Small summary (high signal)
schema_summary_df = pd.DataFrame([{
    "train_rows": df_raw_train_data.shape[0],
    "test_rows": df_raw_test_data.shape[0],
    "train_cols": df_raw_train_data.shape[1],
    "test_cols": df_raw_test_data.shape[1],
    "train_only_count": int((combined_schema_raw["present_in_train"] & ~combined_schema_raw["present_in_test"]).sum()),
    "test_only_count": int((~combined_schema_raw["present_in_train"] & combined_schema_raw["present_in_test"]).sum()),
    "dtype_mismatch_count": int(
        (combined_schema_raw["present_in_train"]
         & combined_schema_raw["present_in_test"]
         & combined_schema_raw["dtype_mismatch"]).sum()
    ),
}])

display(schema_summary_df)
log(f"[raw][schema] notebook_summary: {schema_summary_df.to_dict(orient='records')[0]}")

# Delta-only schema view (columns present in only one split OR dtype mismatch)
schema_deltas_df = combined_schema_raw.loc[
    (combined_schema_raw["present_in_train"] ^ combined_schema_raw["present_in_test"])
    | (
        combined_schema_raw["present_in_train"]
        & combined_schema_raw["present_in_test"]
        & combined_schema_raw["dtype_mismatch"]
    ),
    ["column_name", "train_dtype", "test_dtype", "present_in_train", "present_in_test", "dtype_mismatch"],
].copy()

schema_deltas_df = (
    schema_deltas_df
    .sort_values(["dtype_mismatch", "column_name"], ascending=[False, True])
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(schema_deltas_df)

log(f"[raw][schema] delta_rows={schema_deltas_df.shape[0]}")

,train_rows,test_rows,train_cols,test_cols,train_only_count,test_only_count,dtype_mismatch_count
0,466285,421094,75,74,1,0,1


,column_name,train_dtype,test_dtype,present_in_train,present_in_test,dtype_mismatch
0,verification_status_joint,float64,str,True,True,True
1,Unnamed: 0,int64,NaN,True,False,False


In [6]:
# ------------------------------------------------------------------
# Feature universe overview (raw train vs raw test)
# For Appendix A
# ------------------------------------------------------------------

inspection_summary_train = initial_inspection(df_raw_train_data, PROJECT_LOG_FILE)
inspection_summary_test = initial_inspection(df_raw_test_data, PROJECT_LOG_FILE)

feature_summary_train_df = inspection_summary_train["feature_summary"].copy()
feature_summary_test_df = inspection_summary_test["feature_summary"].copy()

train_overview_df = (
    feature_summary_train_df
    .reset_index()
    .rename(columns={
        "index": "column_name",
        "dtype": "train_dtype_inferred",
        "n_unique": "train_n_unique",
        "null_percent": "train_null_percent",
        "is_numeric": "train_is_numeric",
        "is_object": "train_is_object",
        "is_mixed_type": "train_is_mixed_type",
        "is_numeric_object": "train_is_numeric_object",
        "is_fully_null": "train_is_fully_null",
        "is_constant": "train_is_constant",
    })
)

test_overview_df = (
    feature_summary_test_df
    .reset_index()
    .rename(columns={
        "index": "column_name",
        "dtype": "test_dtype_inferred",
        "n_unique": "test_n_unique",
        "null_percent": "test_null_percent",
        "is_numeric": "test_is_numeric",
        "is_object": "test_is_object",
        "is_mixed_type": "test_is_mixed_type",
        "is_numeric_object": "test_is_numeric_object",
        "is_fully_null": "test_is_fully_null",
        "is_constant": "test_is_constant",
    })
)

combined_feature_universe_df = pd.merge(
    train_overview_df,
    test_overview_df,
    on="column_name",
    how="outer",
)

combined_feature_universe_df["max_null_percent"] = combined_feature_universe_df[
    ["train_null_percent", "test_null_percent"]
].max(axis=1)

combined_feature_universe_df["null_gap_test_minus_train"] = (
    combined_feature_universe_df["test_null_percent"] - combined_feature_universe_df["train_null_percent"]
)

combined_feature_universe_df = pd.merge(
    combined_feature_universe_df,
    combined_schema_raw[["column_name", "present_in_train", "present_in_test", "train_dtype", "test_dtype", "dtype_mismatch"]],
    on="column_name",
    how="left",
)

# Severity sorting: mismatches + missingness + structural red flags first
combined_feature_universe_df["severity"] = 0
combined_feature_universe_df.loc[combined_feature_universe_df["dtype_mismatch"] == True, "severity"] += 100
combined_feature_universe_df.loc[
    (combined_feature_universe_df["present_in_train"] ^ combined_feature_universe_df["present_in_test"]) == True,
    "severity"
] += 80
combined_feature_universe_df.loc[
    (combined_feature_universe_df["train_is_fully_null"] == True) | (combined_feature_universe_df["test_is_fully_null"] == True),
    "severity"
] += 60
combined_feature_universe_df.loc[
    (combined_feature_universe_df["train_is_constant"] == True) | (combined_feature_universe_df["test_is_constant"] == True),
    "severity"
] += 40
combined_feature_universe_df.loc[combined_feature_universe_df["max_null_percent"] >= 50, "severity"] += 10

combined_feature_universe_df = (
    combined_feature_universe_df
    .sort_values(["severity", "max_null_percent", "column_name"], ascending=[False, False, True])
    .drop(columns=["severity"])
    .reset_index(drop=True)
)

log(f"[raw][universe] rows={combined_feature_universe_df.shape[0]} cols={combined_feature_universe_df.shape[1]}")


#-------------------------------------------------------------------------
# Actionable insights: columns with dtype mismatches, 
# presence mismatches, high nulls, or constant values
#-------------------------------------------------------------------------

action_df = combined_feature_universe_df.loc[
    (combined_feature_universe_df["dtype_mismatch"] == True)
    | ((combined_feature_universe_df["present_in_train"] ^ combined_feature_universe_df["present_in_test"]) == True)
    | (combined_feature_universe_df["max_null_percent"] >= 50)
    | (combined_feature_universe_df["train_is_constant"] == True)
    | (combined_feature_universe_df["test_is_constant"] == True),
    [
        "column_name",
        "present_in_train",
        "present_in_test",
        "train_dtype",
        "test_dtype",
        "dtype_mismatch",
        "train_null_percent",
        "test_null_percent",
        "max_null_percent",
        "null_gap_test_minus_train",
        "train_is_fully_null",
        "test_is_fully_null",
        "train_is_constant",
        "test_is_constant",
    ],
].copy()

action_df = action_df.sort_values(
    ["dtype_mismatch", "max_null_percent", "column_name"],
    ascending=[False, False, True],
).reset_index(drop=True)

display(action_df)
log(f"[raw][universe] action_rows={action_df.shape[0]}")

,column_name,present_in_train,present_in_test,train_dtype,test_dtype,dtype_mismatch,train_null_percent,test_null_percent,max_null_percent,null_gap_test_minus_train,train_is_fully_null,test_is_fully_null,train_is_constant,test_is_constant
0,verification_status_joint,True,True,float64,str,True,100.00,99.88,100.00,-0.12,True,False,False,False
1,all_util,True,True,float64,float64,False,100.00,94.92,100.00,-5.08,True,False,False,False
2,annual_inc_joint,True,True,float64,float64,False,100.00,99.88,100.00,-0.12,True,False,False,False
3,dti_joint,True,True,float64,float64,False,100.00,99.88,100.00,-0.12,True,False,False,False
4,il_util,True,True,float64,float64,False,100.00,95.58,100.00,-4.42,True,False,False,False
5,inq_fi,True,True,float64,float64,False,100.00,94.92,100.00,-5.08,True,False,False,False
6,inq_last_12m,True,True,float64,float64,False,100.00,94.92,100.00,-5.08,True,False,False,False
7,max_bal_bc,True,True,float64,float64,False,100.00,94.92,100.00,-5.08,True,False,False,False
8,mths_since_rcnt_il,True,True,float64,float64,False,100.00,95.06,100.00,-4.94,True,False,False,False
9,open_acc_6m,True,True,float64,float64,False,100.00,94.92,100.00,-5.08,True,False,False,False


In [7]:
# -----------------------------------------
# Appendix export: Raw feature universe
# -----------------------------------------

combined_feature_universe_export_df = combined_feature_universe_df.drop(
    columns=["train_dtype_inferred", "test_dtype_inferred"],
    errors="ignore",
).copy()

combined_feature_universe_export_df["train_dtype"] = combined_feature_universe_export_df["train_dtype"].astype("string")
combined_feature_universe_export_df["test_dtype"] = combined_feature_universe_export_df["test_dtype"].astype("string")

combined_feature_universe_export_df.to_csv(appendix_csv_file, index=False, encoding="utf-8")

log(f"[raw][appendix] csv_saved={appendix_csv_file}")

{
    "stage": "appendix_raw_feature_universe",
    "file_saved": True,
    "path": str(appendix_csv_file),
}

{'stage': 'appendix_raw_feature_universe',
 'file_saved': True,
 'path': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\audit\\appendix_A_raw_feature_universe.csv'}

In [8]:
# Creating a structural issues report for the raw feature universe
structural_issues_df = build_structural_issues_report(
    combined_feature_universe_df=combined_feature_universe_df,
    log=log,
)

# Structural issues summary (high signal)
issue_counts_df = (
    structural_issues_df
    .groupby(["issue"], as_index=False)
    .size()
    .rename(columns={"size": "count"})
    .sort_values(["count", "issue"], ascending=[False, True])
    .reset_index(drop=True)
)

display(issue_counts_df)
log(f"[raw][structural_issues] issue_counts={issue_counts_df.to_dict(orient='records')}")

,issue,count
0,High Missing (>50%),24
1,100% Null,17
2,Constant,3
3,Present in train only,1
4,Dtype mismatch,1


In [9]:
# For each identified issue type, we can review example columns to understand the specific problems and potential root causes. This can inform our cleaning and preprocessing strategy in the next steps.
issue_priority = [
    "Present in train only",
    "Present in test only",
    "Dtype mismatch",
    "100% Null",
    "High Missing (>50%)",
    "Constant",
]

examples_df = (
    structural_issues_df
    .assign(issue=pd.Categorical(structural_issues_df["issue"], categories=issue_priority, ordered=True))
    .sort_values(["issue", "column_name", "applies_to"])
    .groupby("issue", as_index=False)
    .head(5)
    .reset_index(drop=True)
)

display(examples_df)
log(f"[raw][structural_issues] example_rows={examples_df.shape[0]}")

,column_name,issue,applies_to
0,Unnamed: 0,Present in train only,train
1,verification_status_joint,Dtype mismatch,both
2,all_util,100% Null,train
3,annual_inc_joint,100% Null,train
4,dti_joint,100% Null,train
5,il_util,100% Null,train
6,inq_fi,100% Null,train
7,all_util,High Missing (>50%),test
8,annual_inc_joint,High Missing (>50%),test
9,desc,High Missing (>50%),test


In [10]:
# Mapping from issue type to recommended action for cleaning/preprocessing
action_map = {
    "Present in train only": "Drop ingestion artifact / align schema",
    "Present in test only": "Align schema (investigate source) / drop if absent in train",
    "Dtype mismatch": "Coerce to consistent dtype (string/category/numeric)",
    "100% Null": "Drop (non-informative in this cohort)",
    "High Missing (>50%)": "Flag as regime-shift sensitive; likely drop or isolate",
    "Constant": "Drop (non-informative)",
}

decision_queue_df = (
    structural_issues_df
    .assign(recommended_action=structural_issues_df["issue"].map(action_map))
    .drop_duplicates(subset=["column_name", "issue", "applies_to"])
    .sort_values(["issue", "column_name", "applies_to"])
    .reset_index(drop=True)
)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.max_colwidth", None,
    "display.width", None,
):
    display(decision_queue_df.head(25))
log(f"[raw][structural_issues] decision_queue_rows={decision_queue_df.shape[0]}")

,column_name,issue,applies_to,recommended_action
0,Unnamed: 0,Present in train only,train,Drop ingestion artifact / align schema
1,verification_status_joint,Dtype mismatch,both,Coerce to consistent dtype (string/category/numeric)
2,all_util,100% Null,train,Drop (non-informative in this cohort)
3,annual_inc_joint,100% Null,train,Drop (non-informative in this cohort)
4,dti_joint,100% Null,train,Drop (non-informative in this cohort)
5,il_util,100% Null,train,Drop (non-informative in this cohort)
6,inq_fi,100% Null,train,Drop (non-informative in this cohort)
7,inq_last_12m,100% Null,train,Drop (non-informative in this cohort)
8,max_bal_bc,100% Null,train,Drop (non-informative in this cohort)
9,mths_since_rcnt_il,100% Null,train,Drop (non-informative in this cohort)


In [11]:
reports = build_string_alignment_report(
	training_dataframe=df_raw_train_data,
	test_dataframe=df_raw_test_data,
	sample_size=5,
	top_k=30,
	drilldown_max_columns=10,
	drilldown_top_values_per_side=20,
	log_file=PROJECT_LOG_FILE,
	export_dir=None,
)

display(reports["summary"])
display(reports["top_deltas"])
display(reports["value_differences"])

,string_like_cols_train,string_like_cols_test,string_like_cols_union,string_like_cols_with_differences,top_k_returned
0,22,23,23,16,16


,column_name,dtype_train,unique_including_null_train,unique_non_null_train,null_percent_train,sample_values_train,dtype_test,unique_including_null_test,unique_non_null_test,null_percent_test,sample_values_test,present_in_train,present_in_test,dtype_mismatch,unique_non_null_gap_test_minus_train,null_gap_test_minus_train,max_null_percent,has_difference
0,verification_status_joint,NaN,NaN,NaN,NaN,NaN,str,4,3,99.88,"[Not Verified, Verified, Source Verified]",False,True,False,3.0,NaN,99.88,True
1,next_pymnt_d,str,101.0,100.0,48.73,"[Feb-16, Jan-16, Sep-13, Feb-14, May-14]",str,4,3,6.12,"[Jan-16, Feb-16, Mar-16]",True,True,False,-97.0,-42.61,48.73,True
2,desc,str,124436.0,124435.0,72.98,[Borrower added on 12/22/11 > I need to upgrad...,str,35,34,99.99,[We knew that using our credit cards to financ...,True,True,False,-124401.0,27.01,99.99,True
3,last_pymnt_d,str,99.0,98.0,0.08,"[Jan-15, Apr-13, Jun-14, Jan-16, Apr-12]",str,14,13,4.10,"[Jan-16, Dec-15, Nov-15, Oct-15, Sep-15]",True,True,False,-85.0,4.02,4.10,True
4,emp_title,str,205476.0,205475.0,5.92,"[Ryder, AIR RESOURCES BOARD, University Medica...",str,120812,120811,5.67,"[Foreign Service Officer, Associate Consultant...",True,True,False,-84664.0,-0.25,5.92,True
5,title,str,63099.0,63098.0,0.00,"[Computer, bike, real estate business, persone...",str,28,27,0.03,"[Home improvement, Credit card refinancing, De...",True,True,False,-63071.0,0.03,0.03,True
6,last_credit_pull_d,str,104.0,103.0,0.01,"[Jan-16, Sep-13, Jan-15, Sep-15, Dec-14]",str,15,14,0.00,"[Jan-16, Dec-15, Nov-15, Sep-15, Oct-15]",True,True,False,-89.0,-0.01,0.01,True
7,url,str,466285.0,466285.0,0.00,[https://www.lendingclub.com/browse/loanDetail...,str,421094,421094,0.00,[https://www.lendingclub.com/browse/loanDetail...,True,True,False,-45191.0,0.00,0.00,True
8,issue_d,str,91.0,91.0,0.00,"[Dec-11, Nov-11, Oct-11, Sep-11, Aug-11]",str,12,12,0.00,"[Dec-15, Nov-15, Oct-15, Sep-15, Aug-15]",True,True,False,-79.0,0.00,0.00,True
9,zip_code,str,888.0,888.0,0.00,"[860xx, 309xx, 606xx, 917xx, 972xx]",str,914,914,0.00,"[200xx, 462xx, 672xx, 460xx, 605xx]",True,True,False,26.0,0.00,0.00,True


,column_name,value,present_in
0,verification_status_joint,Not Verified,test_only
1,verification_status_joint,Source Verified,test_only
2,verification_status_joint,Verified,test_only
3,next_pymnt_d,Apr-08,train_only
4,next_pymnt_d,Apr-09,train_only
...,...,...,...
281,zip_code,507xx,test_only
282,zip_code,509xx,test_only
283,zip_code,520xx,test_only
284,zip_code,555xx,test_only


In [12]:
# -----------------------------------------------------------------------------------
# Raw string alignment checkpoint (train vs test)
# Purpose: detect categorical drift, casing/whitespace issues, and dtype quirks
# before any transformations or column drops.
# -----------------------------------------------------------------------------------

alignment_report = build_string_alignment_report(
    training_dataframe=df_raw_train_data,
    test_dataframe=df_raw_test_data,
    sample_size=5,
    top_k=30,
    drilldown_max_columns=10,
    drilldown_top_values_per_side=20,
    log_file=PROJECT_LOG_FILE,
    export_dir=audit_dir,
    export_base_name="string_alignment",
    export_tag="raw",
    export_sample_values_as_json=True,
)

log(
    f"[raw][string_alignment] audit_files_created=True "
    f"export_dir={audit_dir}"
)

{
    "stage": "raw_string_alignment",
    "audit_files_created": True,
    "export_dir": str(audit_dir),
}

{'stage': 'raw_string_alignment',
 'audit_files_created': True,
 'export_dir': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\audit'}

# Raw Layer – Structural Governance and Eligibility

The data is provided as two independent source files:

- 2007–2014 vintage (train)
- 2015 vintage (test)

The training vintage defines the admissible feature universe.  
The testing vintage is aligned to it.

> **Note:** Governance decisions below are based on the full audit outputs stored in `artifacts/audit` and the LendingClub Data Dictionary accompanying the dataset. Variable eligibility and lifecycle interpretation follow the documented field definitions.

---

## 1. Initial Data Inspection – Structural Baseline

| Metric | Train (2007–2014) | Test (2015) |
|--------|-------------------|-------------|
| Rows | **466,285** | **421,094** |
| Columns | **75** | **74** |
| Memory Usage | ~389 MB | ~323 MB |
| Numeric Columns | 49 | 51 |
| Object / String Columns | 26 | 23 |
| Fully Null Columns | **17** | 0 |
| Constant Columns | 2 (`policy_code`, `application_type`) | 1 (`policy_code`) |
| Columns >50% Null | 4 | Several (overlapping high-missing fields) |
| String-like Columns (post dtype scan) | 22 | 23 |
| Columns present only in one split | 1 (`Unnamed: 0`, train only) | 0 |
| Dtype mismatches | 1 (`verification_status_joint`) | — |

---

### Fully Null Variables (Training-Defined Constraint)

17 columns — approximately **23% of the training feature space** — are entirely null in the training data.
According to the LendingClub Data Dictionary, these fields represent later-introduced bureau segmentation metrics and joint-application variables, including:

- Short-horizon account opening activity (6–24 month windows)
- Installment-specific balance and utilization measures
- Segmented inquiry counts
- Joint borrower income and verification attributes

Their absence is structural: they were not reported during the period covered by the training data.
In the testing data, these columns remain 94–99% null, indicating limited late-period reporting rather than stable feature availability. This reflects reporting expansion rather than a meaningful shift in information availability.
The model is trained exclusively on the training data. A variable that does not exist during training cannot be estimated or validated. Allowing these fields to appear only in the testing data would introduce features the model has never observed.

All 17 columns are therefore removed from both datasets.

---

### Constant Variables and Structural Artifacts

Two columns — `policy_code` and `application_type` — exhibit zero variance in the training data.

`policy_code` reflects an internal platform designation, and `application_type` indicates whether the application is individual or joint. In the training data, `application_type` is exclusively `INDIVIDUAL`, and `policy_code` does not vary. A variable without variation carries no informational content and cannot contribute to model estimation.
Both columns are therefore removed from the datasets.

`Unnamed: 0` appears only in the training file.
It is an exported index column introduced during CSV generation and does not correspond to a documented field in the Data Dictionary.
It is therefore also removed.

---

### Credit Timing Variables

- `mths_since_last_delinq` (Training: **~53.7%** null | Testing: **~48.4%** null)  
- `mths_since_last_major_derog` (Training: **~78.8%** null | Testing: **~70.9%** null)  
- `mths_since_last_record` (Training: **~86.6%** null | Testing: **~82.3%** null)

These variables measure the number of months since the borrower last experienced a negative credit event.  
This includes missed payments, serious credit impairments, or public records such as bankruptcies or legally recorded unpaid debts.
They capture the recency of adverse credit behavior at the time of application.
A value of `0` does not indicate absence. It indicates that the event occurred within the past month. Smaller numeric values represent more recent adverse events.
Missing values (`NaN`) indicate that no such event has occurred in the borrower’s recorded credit history.
The high null percentages in both training and testing data reflect structural absence — many borrowers have no prior adverse events — rather than incomplete reporting.
Recency of prior credit issues is structurally informative. A borrower with a recent missed payment carries different risk implications than a borrower with no prior credit impairments.
Dropping these variables would discard meaningful signal. Leaving them as `NaN` would conflate structural absence with missing data.

Handling:

1. Create a binary indicator: `has_<variable>`  
2. Replace missing values in the original column with sentinel value `9999`

The sentinel value lies well beyond any realistic observed month count. This preserves ordinal interpretation: larger values indicate more distant events, and the sentinel consistently represents “no prior event” without colliding with valid data.

Absence is encoded explicitly while preserving numeric ordering.

---

## 2. Submission-Time Boundary

Prediction is defined at the moment the borrower submits the application.

The admissible information set therefore consists of:

- Borrower-declared application inputs  
- Credit bureau data retrieved at submission  

No later platform actions, pricing decisions, funding outcomes, or servicing updates are allowed to enter the training feature space.

A variable is eligible only if:

1. It is available at submission  
2. It does not encode underwriting, pricing, funding, or servicing outcomes  
3. It contains usable observations in the training data  

The training data defines the admissible feature universe.  
The testing data is aligned to that universe.  
Additional reporting fields that appear in the testing data but are absent in training are removed to preserve temporal consistency.

Variables are excluded from the modeling feature set when they violate this boundary:

- **Underwriting and pricing outputs** (`grade`, `sub_grade`, `int_rate`, `installment`, `verification_status`) reflect LendingClub’s internal risk assessment.  
  Including them would embed prior decision logic into the model, creating circularity rather than independent inference.

- **Funding and allocation fields** (`funded_amnt`, `funded_amnt_inv`) encode platform or investor response after review.  
  These are not borrower characteristics and would introduce selection bias.

- **Servicing and lifecycle variables** (`total_pymnt*`, `recoveries`, `out_prncp*`, `last_pymnt_*`, etc.) are observed only after issuance.  
  Including them would constitute leakage, as they encode post-origination outcomes.

These variables are removed from the `feature_base` dataset used for training.
They are retained in the `clean` dataset when structurally valid, so that benchmarking, diagnostic comparison, and audit checks remain possible without contaminating the modeling input layer.
This boundary governs model training while preserving contextual information and temporal alignment.

---

## 3. Feature Classification Overview – Application Submission Prediction Point

| Column | Category | Action | Rationale |
|--------|----------|--------|-----------|
| `annual_inc`, `dti` | Application input | Keep | Declared at submission |
| `loan_amnt`, `term`, `purpose` | Application input | Keep | Application-time inputs |
| `home_ownership`, `emp_length` | Application input | Keep | Observable at submission |
| `open_acc`, `total_acc` | Credit snapshot | Keep | Credit file state at application |
| `inq_last_6mths` | Credit snapshot | Keep | Recent inquiry activity |
| `delinq_2yrs` | Credit snapshot | Keep | Historical delinquency count |
| `pub_rec` | Credit snapshot | Keep | Public record count |
| `collections_12_mths_ex_med` | Credit snapshot | Keep | Recent collections history |
| `revol_bal`, `revol_util` | Credit snapshot | Keep | Revolving exposure and utilization |
| `acc_now_delinq` | Credit snapshot | Keep | Current delinquency indicator |
| `tot_cur_bal`, `tot_coll_amt` | Credit snapshot | Keep | Aggregate credit balances |
| `mths_since_last_delinq` | Credit timing | Keep (Transformed) | Structural absence encoded explicitly |
| `mths_since_last_major_derog` | Credit timing | Keep (Transformed) | Structural absence encoded explicitly |
| `mths_since_last_record` | Credit timing | Keep (Transformed) | Structural absence encoded explicitly |
| `loan_status` | Target | Target Only | Defines outcome |
| `grade`, `sub_grade` | Platform signal | Benchmark Only | Origination-time assessment |
| `int_rate`, `installment` | Pricing output | Benchmark Only | Determined during underwriting |
| `earliest_cr_line` | Credit timestamp | Exclude | Cannot derive credit age without submission timestamp |
| `verification_status`, `is_inc_v` | Underwriting outcome | Exclude | Determined after submission |
| `funded_amnt`, `funded_amnt_inv` | Funding decision | Exclude | Platform allocation decision |
| `issue_d` | Lifecycle timestamp | Exclude | Occurs after submission |
| `initial_list_status` | Workflow variable | Exclude | Listing outcome |
| Lifecycle updates (`last_credit_pull_d`, `last_fico_range_*`) | Bureau update | Exclude | Updated after submission |
| Servicing variables (`total_pymnt*`, `recoveries`, `out_prncp*`, `last_pymnt_*`, `pymnt_plan`) | Post-origination | Exclude | Observed after issuance |
| Structurally null 2007–2014 variables | Vintage-dependent | Drop | No training observations |
| Identifiers (`id`, `member_id`, `url`, `Unnamed: 0`) | Structural | Drop | Non-predictive artifacts |
| Constants (`policy_code`, `application_type`) | Structural | Drop | Zero variance |
| Free-text (`desc`, `emp_title`, `title`) | Unstructured | Drop | High cardinality; unstructured; Outside structured scope |
| Geographic proxies (`addr_state`, `zip_code`) | Application input | Drop | Removed to reduce proxy discrimination risk |

<br>
<br>

> **Future enhancement:**  
If a verified application submission timestamp becomes available, `earliest_cr_line` can be converted into `credit_age_years` and reconsidered under the temporal availability rule.

---

## 4. String Column Normalization

After structural removal and boundary classification, the remaining object-based fields require normalization.
These columns represent borrower-declared categories, platform classifications retained for benchmarking, and lifecycle timestamps that remain excluded from modeling.

Normalization ensures:

- Consistent representation across training and testing data  
- Stable typing (categorical, numeric, datetime)  
- Removal of formatting variation without altering meaning  

This step standardizes representation only.  
Feature eligibility is governed in the previous section.

The transformations below are applied consistently across both datasets.

---

| Column | Category | Transformation Action | Training Eligibility | Rationale |
|---------|------------|------------------------|----------------------|------------|
| `term` | Structured categorical | Strip whitespace → extract numeric term (36 / 60) → rename to `term_months` → convert to int | Keep | Contractual term declared at submission |
| `emp_length` | Ordinal categorical | Map to integer scale 0–10 (`<1` → 0, `10+` → 10); retain NaN | Keep | Missing indicates unknown tenure, not structural absence |
| `home_ownership` | Categorical | Standardize casing; map `NONE` → `OTHER`; normalize representation (strip/case/mapping) | Keep | Borrower-declared attribute |
| `purpose` | Categorical | Standardize casing; normalize representation (strip/case/mapping) | Keep | Borrower-declared loan purpose |
| `grade` | Platform risk signal | Standardize casing; normalize representation (strip/case/mapping) | Retain (Benchmark Only) | Assigned during underwriting |
| `sub_grade` | Platform risk signal | Standardize casing; normalize representation (strip/case/mapping) | Retain (Benchmark Only) | Granular underwriting signal |
| `verification_status` | Underwriting outcome | Standardize casing; normalize representation (strip/case/mapping) | Exclude | Determined during review process |
| `initial_list_status` | Workflow variable | Standardize casing; normalize representation (strip/case/mapping) | Exclude | Platform listing outcome |
| `pymnt_plan` | Servicing indicator | Map `n` → 0, `y` → 1 | Exclude | Post-origination servicing flag |
| `issue_d` | Lifecycle timestamp | Convert month-year string to datetime | Exclude | Occurs after submission |
| `last_credit_pull_d` | Lifecycle update | Convert month-year string to datetime | Exclude | Subsequent bureau update |
| `last_pymnt_d` | Servicing timeline | Convert month-year string to datetime | Exclude | Post-origination variable |
| `next_pymnt_d` | Servicing timeline | Convert month-year string to datetime | Exclude | Post-origination variable |
| `loan_status` | Target | No transformation at this stage | Keep (Target Only) | Outcome definition; cohort defined later |

---

## 5. Transformation Execution

The following sequence implements the governance rules defined above.

Execution steps:

- Create a stable internal identifier `row_id` during transformation so rows can be traced across the governed outputs (`clean` and `feature_base`) and any downstream audit or evaluation artifacts.
- Remove structurally ineligible columns (fully null in training data, constants, identifiers, free-text fields, geographic proxies, and post-origination variables).
- Align testing data to the training-defined column universe before type normalization.
- Apply credit-timing encoding (`has_*` indicators + `9999` sentinel) for the `mths_since_last_*` variables.
- Normalize remaining string fields using the column-level rules defined in Section 4.
- Convert month-year strings to datetime for lifecycle fields (structural consistency only; these remain excluded from modeling).

Outputs are persisted as:

- `clean` — governed and normalized dataset; benchmark variables retained but not eligible for training.
- `feature_base` — training-eligible predictors plus target; leakage and post-submission fields removed.

In [13]:
# Create row identifier for training data
df_clean_train_data = create_row_identifier(
    df_raw_train_data,
    id_column_name="row_id",
    log_file=PROJECT_LOG_FILE
)

# Create row identifier for test data
df_clean_test_data = create_row_identifier(
    df_raw_test_data,
    id_column_name="row_id",
    log_file=PROJECT_LOG_FILE
)

train_unique = df_clean_train_data["row_id"].is_unique
test_unique = df_clean_test_data["row_id"].is_unique

assert train_unique, "row_id not unique in training data"
assert test_unique, "row_id not unique in test data"

log(
    "[row_id_created] "
    f"train_rows={df_clean_train_data.shape[0]} "
    f"test_rows={df_clean_test_data.shape[0]} "
    f"train_unique={train_unique} "
    f"test_unique={test_unique}"
)

{
    "stage": "row_id_created",
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
    "train_unique": train_unique,
    "test_unique": test_unique,
}

{'stage': 'row_id_created',
 'train_rows': 466285,
 'test_rows': 421094,
 'train_unique': True,
 'test_unique': True}

In [14]:
expected_train_rows = df_clean_train_data.shape[0]
expected_test_rows = df_clean_test_data.shape[0]

{
    "stage": "row_count_baseline_recorded",
    "expected_train_rows": int(expected_train_rows),
    "expected_test_rows": int(expected_test_rows),
}

{'stage': 'row_count_baseline_recorded',
 'expected_train_rows': 466285,
 'expected_test_rows': 421094}

In [15]:
# Log sanity checks on the row_id column in both datasets
log_dataframe_checkpoint(
    df_clean_train_data,
    dataset_name="train",
    checkpoint_name="row_id_created",
    log=log,
    id_column_name="row_id",
)

log_dataframe_checkpoint(
    df_clean_test_data,
    dataset_name="test",
    checkpoint_name="row_id_created",
    log=log,
    id_column_name="row_id",
)

In [16]:
# Parameters for sentinel encoding of credit history features.
CREDIT_RECENCY_COLUMNS = [
    "mths_since_last_delinq",
    "mths_since_last_record",
    "mths_since_last_major_derog"
]

SENTINEL_VALUE = 9999

# Columns to drop for clean_datasets
STRUCTURAL_DROP_COLUMNS = [
    "annual_inc_joint", "dti_joint", "verification_status_joint", "open_acc_6m", "open_il_6m", "open_il_12m", 
    "open_il_24m","mths_since_rcnt_il", "total_bal_il", "il_util","open_rv_12m", "open_rv_24m", "all_util",
    "inq_fi", "total_cu_tl", "inq_last_12m", "max_bal_bc","id", "member_id","url", "Unnamed: 0", "policy_code", 
    "application_type", "desc", "emp_title", "title", "addr_state", "zip_code"
]

#columns to drop for modeling datasets (after cleaning)
EXCLUDED_COLUMNS = [
    "verification_status", "funded_amnt", "funded_amnt_inv", "issue_d", "initial_list_status",
    "last_credit_pull_d", "total_pymnt", "total_pymnt_inv", "total_rec_prncp", "total_rec_int", 
    "total_rec_late_fee", "recoveries", "collection_recovery_fee", "earliest_cr_line", "out_prncp", 
    "out_prncp_inv", "last_pymnt_d", "next_pymnt_d", "last_pymnt_amnt", "pymnt_plan"
]

BENCHMARK_COLUMNS = ["grade", "sub_grade", "int_rate", "installment"]



In [17]:
# Apply sentinel encoding to the credit recency columns in the training data
df_clean_train_data = apply_credit_sentinel_encoding(
    df_clean_train_data,
    CREDIT_RECENCY_COLUMNS,
    sentinel_value=SENTINEL_VALUE,
    log_file=PROJECT_LOG_FILE
)

# Apply sentinel encoding to the credit recency columns in the test data
df_clean_test_data = apply_credit_sentinel_encoding(
    df_clean_test_data,
    CREDIT_RECENCY_COLUMNS,
    sentinel_value=SENTINEL_VALUE,
    log_file=PROJECT_LOG_FILE
)

credit_recency_indicator_columns = [f"has_{column_name}" for column_name in CREDIT_RECENCY_COLUMNS]

train_indicators_present = all(
    column_name in df_clean_train_data.columns
    for column_name in credit_recency_indicator_columns
)
test_indicators_present = all(
    column_name in df_clean_test_data.columns
    for column_name in credit_recency_indicator_columns
)

assert train_indicators_present, "Credit recency indicator columns missing in training data after sentinel encoding."
assert test_indicators_present, "Credit recency indicator columns missing in test data after sentinel encoding."

{
    "stage": "credit_recency_encoded",
    "columns_processed": list(CREDIT_RECENCY_COLUMNS),
    "indicator_columns_created": list(credit_recency_indicator_columns),
    "train_indicators_present": train_indicators_present,
    "test_indicators_present": test_indicators_present,
    "sentinel_value": SENTINEL_VALUE,
}

{'stage': 'credit_recency_encoded',
 'columns_processed': ['mths_since_last_delinq',
  'mths_since_last_record',
  'mths_since_last_major_derog'],
 'indicator_columns_created': ['has_mths_since_last_delinq',
  'has_mths_since_last_record',
  'has_mths_since_last_major_derog'],
 'train_indicators_present': True,
 'test_indicators_present': True,
 'sentinel_value': 9999}

In [18]:
# Drop columns for cleaning training dataset
df_clean_train_data = drop_columns_with_logging(
    dataframe=df_clean_train_data,
    columns_to_drop=STRUCTURAL_DROP_COLUMNS,
    dataset_name="train_clean",
    log_file=PROJECT_LOG_FILE,
    errors="ignore",
)

# Drop columns for cleaning test dataset
df_clean_test_data = drop_columns_with_logging(
    dataframe=df_clean_test_data,
    columns_to_drop=STRUCTURAL_DROP_COLUMNS,
    dataset_name="test_clean",
    log_file=PROJECT_LOG_FILE,
    errors="ignore",
)

{
    "stage": "structural_columns_dropped",
    "columns_requested": len(STRUCTURAL_DROP_COLUMNS),
    "train_columns_after": int(df_clean_train_data.shape[1]),
    "test_columns_after": int(df_clean_test_data.shape[1]),
}

{'stage': 'structural_columns_dropped',
 'columns_requested': 28,
 'train_columns_after': 51,
 'test_columns_after': 51}

In [19]:
# -----------------------------------
# Columns grouped by transformation type
# -----------------------------------

# 1. Text normalization (strip + lowercase + cast to category)
# Used to standardize categorical values and ensure train/test alignment.
categorical_normalization_columns = [
    "home_ownership",
    "purpose",
    "grade",
    "sub_grade",
    "verification_status",
    "initial_list_status",
    "loan_status",
]

categorical_columns_to_cast = [
    "purpose",
    "home_ownership",
    "verification_status",
    "initial_list_status",
    "grade",
    "sub_grade",
    "loan_status",
]

# 2. Deterministic category remapping
# Used to consolidate semantically equivalent labels.
special_categorical_mappings = {
    "home_ownership": {
        "none": "other",
        "any": "other",
    }
}

# 3. Ordinal encoding
# Used for ordered categories with intrinsic rank.
ordinal_encoding_columns = {
    "emp_length": {
        "< 1 year": 0,
        "1 year": 1,
        "2 years": 2,
        "3 years": 3,
        "4 years": 4,
        "5 years": 5,
        "6 years": 6,
        "7 years": 7,
        "8 years": 8,
        "9 years": 9,
        "10+ years": 10
    }
}

# 4. Structured extraction
# Convert contract term to numeric month representation.
term_column = "term"
term_new_name = "term_months"

# 5. Binary encoding
# Convert binary categorical indicators to numeric form.
binary_encoding_columns = {
    "pymnt_plan": {
        "n": 0,
        "y": 1
    }
}

# 6. Month-Year datetime conversion
# Convert month-year strings to datetime format.
month_year_datetime_columns = [
    "issue_d",
    "last_credit_pull_d",
    "last_pymnt_d",
    "next_pymnt_d",
    "earliest_cr_line",
]

# 7. Explicit mapping for listing type
# Replace abbreviated codes with semantic labels.
initial_list_status_mapping = {
    "w": "whole",
    "f": "fractional"
}

{
    "stage": "transformation_configuration_defined",
    "categorical_columns": len(categorical_normalization_columns),
    "ordinal_columns": len(ordinal_encoding_columns),
    "binary_columns": len(binary_encoding_columns),
    "datetime_columns": len(month_year_datetime_columns),
}


{'stage': 'transformation_configuration_defined',
 'categorical_columns': 7,
 'ordinal_columns': 1,
 'binary_columns': 1,
 'datetime_columns': 5}

In [20]:
# 1. Normalize text categories (casing/whitespace)
df_clean_train_data = normalize_text_columns(
    dataframe=df_clean_train_data,
    columns_to_normalize=categorical_normalization_columns,
    log_file=PROJECT_LOG_FILE,
)

df_clean_test_data = normalize_text_columns(
    dataframe=df_clean_test_data,
    columns_to_normalize=categorical_normalization_columns,
    log_file=PROJECT_LOG_FILE,
)

assert df_clean_train_data.shape[0] > 0
assert df_clean_test_data.shape[0] > 0

{
    "stage": "categorical_text_normalized",
    "columns_processed": list(categorical_normalization_columns),
    "train_columns": int(df_clean_train_data.shape[1]),
    "test_columns": int(df_clean_test_data.shape[1]),
}

{'stage': 'categorical_text_normalized',
 'columns_processed': ['home_ownership',
  'purpose',
  'grade',
  'sub_grade',
  'verification_status',
  'initial_list_status',
  'loan_status'],
 'train_columns': 51,
 'test_columns': 51}

In [21]:
# 2. Home ownership alignment
df_clean_train_data = normalize_home_ownership(
    df_clean_train_data,
    log_file=PROJECT_LOG_FILE,
)

df_clean_test_data = normalize_home_ownership(
    df_clean_test_data,
    log_file=PROJECT_LOG_FILE,
)

assert "home_ownership" in df_clean_train_data.columns
assert "home_ownership" in df_clean_test_data.columns

{
    "stage": "home_ownership_normalized",
    "column": "home_ownership",
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'home_ownership_normalized',
 'column': 'home_ownership',
 'train_rows': 466285,
 'test_rows': 421094}

In [22]:
# 3. initial_list_status mapping (w/f -> words)
df_clean_train_data = apply_categorical_mapping(
    dataframe=df_clean_train_data,
    column_name="initial_list_status",
    mapping=initial_list_status_mapping,
    log_file=PROJECT_LOG_FILE,
    allow_unmapped_values=False,
)

df_clean_test_data = apply_categorical_mapping(
    dataframe=df_clean_test_data,
    column_name="initial_list_status",
    mapping=initial_list_status_mapping,
    log_file=PROJECT_LOG_FILE,
    allow_unmapped_values=False,
)

assert "initial_list_status" in df_clean_train_data.columns
assert "initial_list_status" in df_clean_test_data.columns

{
    "stage": "initial_list_status_mapped",
    "column": "initial_list_status",
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'initial_list_status_mapped',
 'column': 'initial_list_status',
 'train_rows': 466285,
 'test_rows': 421094}

In [23]:
# 4. pymnt_plan binary
df_clean_train_data = apply_categorical_mapping(
    dataframe=df_clean_train_data,
    column_name="pymnt_plan",
    mapping=binary_encoding_columns["pymnt_plan"],
    log_file=PROJECT_LOG_FILE,
    allow_unmapped_values=False,
)

df_clean_test_data = apply_categorical_mapping(
    dataframe=df_clean_test_data,
    column_name="pymnt_plan",
    mapping=binary_encoding_columns["pymnt_plan"],
    log_file=PROJECT_LOG_FILE,
    allow_unmapped_values=False,
)

assert "pymnt_plan" in df_clean_train_data.columns
assert "pymnt_plan" in df_clean_test_data.columns

{
    "stage": "pymnt_plan_binary_encoded",
    "column": "pymnt_plan",
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'pymnt_plan_binary_encoded',
 'column': 'pymnt_plan',
 'train_rows': 466285,
 'test_rows': 421094}

In [24]:
# 5. Term parsing + rename
df_clean_train_data = parse_term_column(
    dataframe=df_clean_train_data,
    term_column=term_column,
    new_column_name=term_new_name,
    log_file=PROJECT_LOG_FILE,
)

df_clean_test_data = parse_term_column(
    dataframe=df_clean_test_data,
    term_column=term_column,
    new_column_name=term_new_name,
    log_file=PROJECT_LOG_FILE,
)

assert term_new_name in df_clean_train_data.columns
assert term_new_name in df_clean_test_data.columns

{
    "stage": "term_parsed_to_months",
    "source_column": term_column,
    "new_column": term_new_name,
}

{'stage': 'term_parsed_to_months',
 'source_column': 'term',
 'new_column': 'term_months'}

In [25]:
# 6. emp_length ordinal
df_clean_train_data = transform_emp_length(
    df_clean_train_data,
    log_file=PROJECT_LOG_FILE,
)

df_clean_test_data = transform_emp_length(
    df_clean_test_data,
    log_file=PROJECT_LOG_FILE,
)

assert "emp_length" in df_clean_train_data.columns
assert "emp_length" in df_clean_test_data.columns

{
    "stage": "emp_length_ordinal_encoded",
    "column": "emp_length",
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'emp_length_ordinal_encoded',
 'column': 'emp_length',
 'train_rows': 466285,
 'test_rows': 421094}

In [26]:
# 7. Month-year datetime conversions
df_clean_train_data = convert_month_year_columns_to_datetime(
    dataframe=df_clean_train_data,
    column_names=month_year_datetime_columns,
    log_file=PROJECT_LOG_FILE,
)

df_clean_test_data = convert_month_year_columns_to_datetime(
    dataframe=df_clean_test_data,
    column_names=month_year_datetime_columns,
    log_file=PROJECT_LOG_FILE,
)

for column_name in month_year_datetime_columns:
    assert column_name in df_clean_train_data.columns
    assert column_name in df_clean_test_data.columns

{
    "stage": "month_year_columns_converted_to_datetime",
    "columns_converted": list(month_year_datetime_columns),
    "train_rows": int(df_clean_train_data.shape[0]),
    "test_rows": int(df_clean_test_data.shape[0]),
}

{'stage': 'month_year_columns_converted_to_datetime',
 'columns_converted': ['issue_d',
  'last_credit_pull_d',
  'last_pymnt_d',
  'next_pymnt_d',
  'earliest_cr_line'],
 'train_rows': 466285,
 'test_rows': 421094}

In [27]:
# 8. Drop original term and emp_length columns after feature extraction
df_clean_train_data = drop_columns_with_logging(
    dataframe=df_clean_train_data,
    columns_to_drop=["term", "emp_length"],
    dataset_name="train_clean",
    log_file=PROJECT_LOG_FILE,
    errors="ignore",
)

df_clean_test_data = drop_columns_with_logging(
    dataframe=df_clean_test_data,
    columns_to_drop=["term", "emp_length"],
    dataset_name="test_clean",
    log_file=PROJECT_LOG_FILE,
    errors="ignore",
)

assert "term" not in df_clean_train_data.columns
assert "term" not in df_clean_test_data.columns
assert "emp_length" not in df_clean_train_data.columns
assert "emp_length" not in df_clean_test_data.columns

{
    "stage": "original_term_and_emp_length_dropped",
    "columns_dropped": ["term", "emp_length"],
    "train_columns_after": int(df_clean_train_data.shape[1]),
    "test_columns_after": int(df_clean_test_data.shape[1]),
}

{'stage': 'original_term_and_emp_length_dropped',
 'columns_dropped': ['term', 'emp_length'],
 'train_columns_after': 51,
 'test_columns_after': 51}

In [28]:
# 9. Standardize categorical dtypes
df_clean_train_data, df_clean_test_data = cast_categorical_columns(
    dataframe_train=df_clean_train_data,
    dataframe_test=df_clean_test_data,
    column_names=categorical_columns_to_cast,
    log_file=PROJECT_LOG_FILE,
)

for column_name in categorical_columns_to_cast:
    assert column_name in df_clean_train_data.columns
    assert column_name in df_clean_test_data.columns

{
    "stage": "categorical_columns_cast_to_category",
    "columns_cast": list(categorical_columns_to_cast),
    "train_columns": int(df_clean_train_data.shape[1]),
    "test_columns": int(df_clean_test_data.shape[1]),
}

{'stage': 'categorical_columns_cast_to_category',
 'columns_cast': ['purpose',
  'home_ownership',
  'verification_status',
  'initial_list_status',
  'grade',
  'sub_grade',
  'loan_status'],
 'train_columns': 51,
 'test_columns': 51}

In [29]:
actual_train_rows = df_clean_train_data.shape[0]
actual_test_rows = df_clean_test_data.shape[0]

if actual_train_rows != expected_train_rows:
    raise ValueError(
        f"Row count drift detected in train dataset. "
        f"expected={expected_train_rows} actual={actual_train_rows}"
    )

if actual_test_rows != expected_test_rows:
    raise ValueError(
        f"Row count drift detected in test dataset. "
        f"expected={expected_test_rows} actual={actual_test_rows}"
    )

{
    "stage": "row_count_invariant_verified",
    "train_rows": int(actual_train_rows),
    "test_rows": int(actual_test_rows),
}

{'stage': 'row_count_invariant_verified',
 'train_rows': 466285,
 'test_rows': 421094}

In [30]:
# -----------------------------------------------------------------------------------
# Clean string alignment checkpoint (train vs test)
# Purpose: detect categorical drift and unexpected value changes
# after transformations and column drops.
# -----------------------------------------------------------------------------------

alignment_report = build_string_alignment_report(
    training_dataframe=df_clean_train_data,
    test_dataframe=df_clean_test_data,
    sample_size=5,
    top_k=30,
    drilldown_max_columns=10,
    drilldown_top_values_per_side=20,
    log_file=PROJECT_LOG_FILE,
    export_dir=audit_dir,
    export_base_name="string_alignment",
    export_tag="clean",
    export_sample_values_as_json=True,
)

log(
    f"[clean][string_alignment] audit_files_created=True "
    f"export_dir={audit_dir}"
)

{
    "stage": "clean_string_alignment",
    "audit_files_created": True,
    "export_dir": str(audit_dir),
}

{'stage': 'clean_string_alignment',
 'audit_files_created': True,
 'export_dir': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\audit'}

In [31]:
# ------------------------------------------------------------------------------
# Create feature_base datasets by removing excluded + benchmark columns
# ------------------------------------------------------------------------------

df_feature_base_train_data = drop_columns_with_logging(
    dataframe=df_clean_train_data,
    columns_to_drop=EXCLUDED_COLUMNS + BENCHMARK_COLUMNS,
    dataset_name="train_feature_base",
    log_file=PROJECT_LOG_FILE,
    errors="raise",
)

df_feature_base_test_data = drop_columns_with_logging(
    dataframe=df_clean_test_data,
    columns_to_drop=EXCLUDED_COLUMNS + BENCHMARK_COLUMNS,
    dataset_name="test_feature_base",
    log_file=PROJECT_LOG_FILE,
    errors="raise",
)

for column_name in EXCLUDED_COLUMNS + BENCHMARK_COLUMNS:
    assert column_name not in df_feature_base_train_data.columns
    assert column_name not in df_feature_base_test_data.columns

{
    "stage": "feature_base_created",
    "dropped_columns": len(EXCLUDED_COLUMNS + BENCHMARK_COLUMNS),
    "train_columns": int(df_feature_base_train_data.shape[1]),
    "test_columns": int(df_feature_base_test_data.shape[1]),
}

{'stage': 'feature_base_created',
 'dropped_columns': 24,
 'train_columns': 27,
 'test_columns': 27}

In [32]:
# -----------------------------------------------------------------------------------
# Feature base string alignment checkpoint (train vs test)
# Purpose: detect categorical drift after feature selection
# -----------------------------------------------------------------------------------

alignment_report = build_string_alignment_report(
    training_dataframe=df_feature_base_train_data,
    test_dataframe=df_feature_base_test_data,
    sample_size=5,
    top_k=30,
    drilldown_max_columns=10,
    drilldown_top_values_per_side=20,
    log_file=PROJECT_LOG_FILE,
    export_dir=audit_dir,
    export_base_name="string_alignment",
    export_tag="feature_base",
    export_sample_values_as_json=True,
)

log(
    f"[feature_base][string_alignment] audit_files_created=True "
    f"export_dir={audit_dir}"
)

{
    "stage": "feature_base_string_alignment",
    "audit_files_created": True,
    "export_dir": str(audit_dir),
}

{'stage': 'feature_base_string_alignment',
 'audit_files_created': True,
 'export_dir': 'D:\\Portfolio\\loans-at-risk-capturing-default\\artifacts\\audit'}

In [33]:
# --------------------------------------------------
# Feature_base schema alignment check
# --------------------------------------------------

train_columns = df_feature_base_train_data.columns.tolist()
test_columns = df_feature_base_test_data.columns.tolist()

columns_only_in_train = sorted(set(train_columns) - set(test_columns))
columns_only_in_test = sorted(set(test_columns) - set(train_columns))

schema_match = not columns_only_in_train and not columns_only_in_test

log(
    "[feature_base][schema_check] "
    f"schema_match={schema_match} "
    f"train_columns={len(train_columns)} "
    f"test_columns={len(test_columns)} "
    f"only_in_train={len(columns_only_in_train)} "
    f"only_in_test={len(columns_only_in_test)}"
)

assert schema_match, (
    "Train/test feature_base columns are not identical.\n"
    f"Only in train: {columns_only_in_train}\n"
    f"Only in test: {columns_only_in_test}"
)

{
    "stage": "feature_base_schema_alignment",
    "schema_match": schema_match,
    "column_count": len(train_columns),
}

{'stage': 'feature_base_schema_alignment',
 'schema_match': True,
 'column_count': 27}

In [34]:
# --------------------------------------------------
# Feature_base dtype alignment check
# --------------------------------------------------

dtype_mismatches = []

for column_name in train_columns:
    train_dtype = df_feature_base_train_data[column_name].dtype
    test_dtype = df_feature_base_test_data[column_name].dtype

    if train_dtype != test_dtype:
        dtype_mismatches.append(
            (column_name, str(train_dtype), str(test_dtype))
        )

assert not dtype_mismatches, (
    "Dtype mismatches detected between train and test feature_base datasets.\n"
    f"First mismatches: {dtype_mismatches[:20]}"
)

log(
    "[feature_base][dtype_check] "
    f"mismatches={len(dtype_mismatches)} "
    f"columns_checked={len(train_columns)}"
)

{
    "stage": "feature_base_dtype_alignment",
    "dtype_mismatches": len(dtype_mismatches),
    "columns_checked": len(train_columns),
}

{'stage': 'feature_base_dtype_alignment',
 'dtype_mismatches': 0,
 'columns_checked': 27}

In [35]:
# --------------------------------------------------
# Save clean and feature_base datasets (parquet)
# --------------------------------------------------

log = get_logger(PROJECT_LOG_FILE)

log(
    "[save_datasets] "
    f"train_clean={df_clean_train_data.shape} | "
    f"test_clean={df_clean_test_data.shape} | "
    f"train_feature_base={df_feature_base_train_data.shape} | "
    f"test_feature_base={df_feature_base_test_data.shape}"
)

log(f"[save_datasets] dataset=train_clean path={clean_train_data_file}")
df_clean_train_data.to_parquet(clean_train_data_file, index=False)

log(f"[save_datasets] dataset=test_clean path={clean_test_data_file}")
df_clean_test_data.to_parquet(clean_test_data_file, index=False)

log(f"[save_datasets] dataset=train_feature_base path={feature_base_train_data_file}")
df_feature_base_train_data.to_parquet(feature_base_train_data_file, index=False)

log(f"[save_datasets] dataset=test_feature_base path={feature_base_test_data_file}")
df_feature_base_test_data.to_parquet(feature_base_test_data_file, index=False)

{
    "stage": "datasets_saved",
    "train_clean_rows": int(df_clean_train_data.shape[0]),
    "test_clean_rows": int(df_clean_test_data.shape[0]),
    "train_feature_base_rows": int(df_feature_base_train_data.shape[0]),
    "test_feature_base_rows": int(df_feature_base_test_data.shape[0]),
}

{'stage': 'datasets_saved',
 'train_clean_rows': 466285,
 'test_clean_rows': 421094,
 'train_feature_base_rows': 466285,
 'test_feature_base_rows': 421094}

# Notebook Conclusion – Data Cleaning & Feature Base Construction

## Objective

Model prediction is defined at **application submission**.

This notebook constructs the transformation layer required to enforce that constraint.

All transformation decisions follow from two conditions:

1. Only information observable at submission may be used for prediction.
2. The training window (2007–2014) defines the learnable feature space.

The goal of this notebook is therefore structural rather than predictive: produce a feature space that can be learned during training and applied at prediction time without leakage.

---

## Structural Boundary Enforcement

Feature eligibility was determined by the submission-time boundary.

The following variables were removed:

- Identifiers and export artifacts  
- Fully null variables within the training window  
- Zero-variance variables  
- Underwriting, pricing, funding, and servicing variables  
- Geographic proxies (`addr_state`, `zip_code`)

Variables absent in the 2007–2014 training data were also removed from both partitions. A model cannot learn from information that does not exist in the training window.

The resulting feature space therefore contains only **borrower-declared and bureau-reported information observable at application submission.**

---

## Categorical and Type Normalization

Object-based variables were normalized and typed explicitly to ensure consistent representation across partitions.

Transformations included:

- Standardization of categorical text values  
- Ordinal encoding of `emp_length` (0–10 scale)  
- Deterministic binary encoding  
- Casting categorical variables to `category` dtype  
- Converting object-encoded numerics to numeric dtype  
- Parsing month–year strings into `datetime64[ns]`

No frequency filtering, target encoding, or derived features were introduced at this stage.

The objective was structural: **identical categorical spaces and datatypes across train and test partitions.**

---

## Vintage-Based Reporting Differences

Three bureau variables exhibit missingness in the training window:

- `tot_coll_amt`
- `tot_cur_bal`
- `total_rev_hi_lim`

Approximately 15% of observations are missing between 2007–2014, while the 2015 test partition contains none.

This difference reflects historical reporting coverage rather than transformation error.

Imputation will therefore be fit exclusively on the training window during modeling, preserving the temporal boundary between training and prediction.

---

## Dataset Construction

Two dataset layers were produced:

**`clean`**

Structurally normalized dataset retaining benchmark variables.

**`feature_base`**

Submission-time eligible variables plus the target.

For both training and test partitions the pipeline verified:

- identical column presence  
- identical column ordering  
- matching datatypes  
- schema integrity  

No mismatches remain.

---

## Final State

The `feature_base` dataset now satisfies the submission-time constraint:

- Only borrower-declared or bureau-reported information remains.  
- No underwriting, pricing, or servicing variables are present.  
- Geographic proxies have been removed.  
- Train and test partitions share identical schema and categorical spaces.  
- Datetime variables are explicitly typed.

Any predictive signal extracted from this dataset must therefore arise from information **available at the time of application.**

---

## Next Phase

The next notebook moves from transformation to analysis.

It will:

- Define the modeling cohort from `loan_status`
- Perform EDA with explicit temporal awareness
- Evaluate feature stability and predictive contribution
- Implement training-fitted imputation and encoding pipelines

The transformation layer now provides a governed and reproducible starting point for modeling.